# SFT: Qwen3.5-0.8B + LoRA on Colab or Kaggle

Runs `lcs train-sft --config configs/sft.yaml` on one GPU and gives you the run directory as a zip:
`adapter/` (LoRA weights + model card), `merged/` (base + adapter as plain weights + card), `metrics.json`, `experiment.json`, `lora.json`.

- **GPU:** any CUDA GPU with 16 GB or more. L4 / A10G / A100 train in bf16. On T4 or P100 (no bf16) the config is switched to fp16 autocast on fp32 weights automatically.
- **Data:** `data/processed/{train,dev}.jsonl` with Claude references, from `lcs prepare-data` and `lcs generate-references` (see `docs/RUNBOOK.md` §1). Mock references train fine but give a model that must not be published.
- **Nothing is pushed** to the Hugging Face Hub. Trackio logs locally; W&B is opt-in.

Run the cells top to bottom (Colab: *Runtime → Change runtime type → GPU*; Kaggle: *Accelerator → GPU*).

In [ ]:
!nvidia-smi

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/erfunm/Legal-Clause-Simplifier-using-RLHF.git"
BRANCH = "main"
CONFIG = "configs/sft.yaml"

# Where data/processed comes from:
#   "upload": a zip you make locally with `zip -r processed.zip data/processed` (Colab only)
#   "kaggle": a Kaggle dataset attached to this notebook that holds train.jsonl, dev.jsonl, ...
#   "hub":    your private dataset from `lcs push-dataset` (needs HF_TOKEN as a secret)
DATA_SOURCE = "upload"
KAGGLE_DATA_DIR = "/kaggle/input/legal-clause-simplification"
HUB_DATASET = "<user>/legal-clause-simplification"

TRACKING = "trackio"  # "trackio" (local, no account), "wandb" (needs the WANDB_API_KEY secret) or "none"

## 2. Install

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} repo
%cd repo
# Keeps the platform's CUDA torch; installs the training stack and trackers on top.
!pip install -q -e ".[train,tracking]"

## 3. Data

In [ ]:
import os
import shutil
import zipfile
from collections import Counter
from pathlib import Path

target = Path("data/processed")
target.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "upload":
    from google.colab import files

    uploaded = files.upload()  # choose processed.zip
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        archive.extractall(".")
elif DATA_SOURCE == "kaggle":
    shutil.copytree(KAGGLE_DATA_DIR, target, dirs_exist_ok=True)
elif DATA_SOURCE == "hub":
    from datasets import load_dataset

    from legal_simplifier.data.schema import ClauseRecord, write_records

    hub = load_dataset(HUB_DATASET)
    for split in hub:
        write_records([ClauseRecord(**row) for row in hub[split]], target / f"{split}.jsonl")
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

from legal_simplifier.data.schema import read_records

train = read_records(target / "train.jsonl")
sources = Counter(r.reference_source for r in train)
print(f"{len(train)} training clauses; reference sources: {dict(sources)}")
if sources.get("mock"):
    print("WARNING: mock references. The pipeline runs, but the model is not a real result.")

## 4. Tracking (optional)

In [ ]:
if TRACKING == "wandb":
    try:
        from google.colab import userdata

        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except ImportError:
        from kaggle_secrets import UserSecretsClient

        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 5. Fit the config to this GPU

In [ ]:
import yaml

from legal_simplifier.config import SFTConfigFile, load_config
from legal_simplifier.training.gpu import adapt_precision

cfg = adapt_precision(load_config(CONFIG, SFTConfigFile))
cfg.tracking.backend = TRACKING
RUNTIME_CONFIG = "configs/runtime_sft.yaml"
Path(RUNTIME_CONFIG).write_text(yaml.safe_dump(cfg.model_dump(mode="json"), sort_keys=False))
print(f"bf16={cfg.training.bf16} fp16={cfg.training.fp16} base dtype={cfg.model.dtype}")
print(f"LoRA r={cfg.lora.r} on {cfg.lora.target_modules}")

## 6. Train

In [ ]:
!lcs train-sft --config {RUNTIME_CONFIG}

## 7. Inspect the run

In [ ]:
import json

from IPython.display import Markdown, display

run_dir = Path(cfg.training.output_dir)
print(json.dumps(json.loads((run_dir / "lora.json").read_text())["parameters"], indent=2))
print(json.dumps(json.loads((run_dir / "metrics.json").read_text()), indent=2))
display(Markdown((run_dir / "adapter" / "README.md").read_text()))

## 8. Download

Copy the zip back into the repo on your machine (`unzip sft_run.zip`), so later stages find `checkpoints/sft/merged/`. On Kaggle, the zip is in `/kaggle/working/repo` under *Output*. Trackio's local dashboard: `trackio show --project legal-clause-simplifier`.

In [ ]:
!zip -qr sft_run.zip {cfg.training.output_dir} -x "*/checkpoint-*"
try:
    from google.colab import files

    files.download("sft_run.zip")
except ImportError:
    print(f"Saved {Path('sft_run.zip').resolve()}")